In [1]:
import pandas as pd

from sklearn import  model_selection
from sklearn.model_selection import RandomizedSearchCV 
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn import tree
from matplotlib import pyplot as plt

from sklearn.model_selection import KFold 
from sklearn.model_selection import cross_val_score 

import numpy as np
import sklearn

from scipy import stats

from sklearn.model_selection import RandomizedSearchCV 

from plotnine import *
import plotnine

from sklearn.metrics import f1_score 

In [6]:
X_treino = pd.read_csv('df5_ins.csv'
                 ,sep = ','
                 ,decimal='.'
                , error_bad_lines=False
                  )

/tmp/ipykernel_261266/2070938832.py:1: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  X_treino = pd.read_csv('df5_ins.csv'


In [25]:
for i in range(0,18):
    
    i=str(i)    
    
    # Extração dos dados
    X_treino = pd.read_csv('df'+i+'_ins.csv'
                     ,sep = ','
                     ,decimal='.'
                     ,on_bad_lines='warn'
                      )


    print('')
    print('Tamanho do treino e quantidade de ocorrências')
    print(len(X_treino.index))
    print(round(sum(X_treino['flg_ocorrencia'])))
    print('')

    X_teste = pd.read_csv('df'+i+'_oos.csv'
                     ,sep = ','
                     ,decimal='.'
                     ,on_bad_lines='warn'
                      )


    print('')
    print('Tamanho do teste e quantidade de ocorrências')
    print(len(X_teste.index))
    print(round(sum(X_teste['flg_ocorrencia'])))
    print('')

    oot = pd.read_csv('df'+i+'_final_oot.csv'
                     ,sep = ','
                     ,decimal='.'
                     ,on_bad_lines='warn'
                      )


    print('')
    print('Tamanho do teste 2 meses a frente e quantidade de ocorrências')
    print(len(oot.index))
    print(round(sum(oot['flg_ocorrencia'])))
    print('')

    #Ajuste na flg_ocorrencia

    y_treino = X_treino['flg_ocorrencia']
    X_treino = X_treino.drop('flg_ocorrencia', axis=1)
    n = len(X_treino.columns)

    y_teste = X_teste['flg_ocorrencia']
    X_teste = X_teste.drop('flg_ocorrencia', axis=1)



    ####################################
    ############ XGboosting ############
    ####################################

    classificador_xgboost = XGBClassifier()

    lista_parametros = {'booster': ['gbtree'],
        'max_delta_step': [i + 1 for i in range(1,11,1)],
        'max_depth': [i + 1 for i in range(2,30,1)],
        'eta': stats.uniform(0, 1),
        'objective' : ['reg:logistic', 'binary:logistic', 'binary:logitraw', 'binary:hinge']
        }

    rand_search = RandomizedSearchCV(classificador_xgboost, 
                                     param_distributions = lista_parametros,
                                     cv = 10,
                                     random_state = 42,
                                     scoring = 'f1') 

    rand_search.fit(X_treino.iloc[:,3:n], y_treino) 

    print('')
    print(rand_search.best_estimator_)
    print('')

    classificador_xgboost = rand_search.best_estimator_
    classificador_xgboost.fit(X_treino.iloc[:,3:n], y_treino) 

    thresholds = np.arange(0.0, 1.0, 0.0001)
    fscore = np.zeros(shape=(len(thresholds)))
    #print('Length of sequence: {}'.format(len(thresholds)))

    # Fit the model
    for index, elem in enumerate(thresholds):
        # Corrected probabilities
        y_pred = classificador_xgboost.predict_proba(X_teste.iloc[:,3:n])[:,1]
        y_pred_prob = (y_pred > elem).astype('int')
        # Calculate the f-score
        fscore[index] = f1_score(y_teste, y_pred_prob)

    # Find the optimal threshold
    index = np.argmax(fscore)
    thresholdOpt = round(thresholds[index], ndigits = 4)
    fscoreOpt = round(fscore[index], ndigits = 4)
    print('Best Threshold: {} with F-Score: {}'.format(thresholdOpt, fscoreOpt))

    #############################################
    ############ Qualidade do modelo ############
    #############################################
    
    kfold  = KFold(n_splits=10, shuffle=False)
    result_xgboost = cross_val_score(classificador_xgboost, X_treino.iloc[:,3:n], y_treino, cv = kfold)
    #K-fold
    #'Média da acurácia:'
    print(np.mean(result_xgboost))
    #'Variância da acurácia'
    print(np.std(result_xgboost))



    #Teste - OOS

    pred_teste = classificador_xgboost.predict_proba(X_teste.iloc[:,3:n])[:,1]
    hardpred_teste_tuned_thresh = np.where(pred_teste >= thresholdOpt, 1, 0)

    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(y_teste, hardpred_teste_tuned_thresh),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(y_teste, hardpred_teste_tuned_thresh),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(y_teste, hardpred_teste_tuned_thresh),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(y_teste, hardpred_teste_tuned_thresh),2))

    if round(sum(oot['flg_ocorrencia'])) <= 0 :
        continue

    #Teste - OOT

    pred_oot=classificador_xgboost.predict_proba(oot.iloc[:,4:n+1])[:,1]
    hardpred_oot_tuned_thresh = np.where(pred_oot >= thresholdOpt, 1, 0)

    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(oot.iloc[:,0], hardpred_oot_tuned_thresh ),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(oot.iloc[:,0], hardpred_oot_tuned_thresh ),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(oot.iloc[:,0], hardpred_oot_tuned_thresh ),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(oot.iloc[:,0], hardpred_oot_tuned_thresh ),2))



/tmp/ipykernel_261266/2109001105.py:6: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


/tmp/ipykernel_261266/2109001105.py:19: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


/tmp/ipykernel_261266/2109001105.py:32: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


/home/pdpa-chuvas/Deslizamento/env/lib/python3.8/site-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 7 members, which is less than n_splits=10.



Tamanho do treino e quantidade de ocorrências
2144
7


Tamanho do teste e quantidade de ocorrências
1164
2


Tamanho do teste 2 meses a frente e quantidade de ocorrências
24869
35



/home/pdpa-chuvas/Deslizamento/env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
/home/pdpa-chuvas/Deslizamento/env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
/home/pdpa-chuvas/Deslizamento/env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
/home/pdpa-chuvas/Deslizamento/env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted s


XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eta=0.3745401188473625, eval_metric=None, feature_types=None,
              gamma=0, gpu_id=-1, grow_policy='depthwise', importance_type=None,
              interaction_constraints='', learning_rate=0.37454012, max_bin=256,
              max_cat_threshold=64, max_cat_to_onehot=4, max_delta_step=9,
              max_depth=23, max_leaves=0, min_child_weight=1, missing=nan,
              monotone_constraints='()', n_estimators=100, n_jobs=0,
              num_parallel_tree=1, objective='binary:logitraw', ...)

Best Threshold: 0.0 with F-Score: 0.6667
0.9981351880026083
0.00228392927142314
1.0
1.0
0.5
0.67
1.0
0.0
0.0
0.0

Tamanho do treino e quantidade de ocorrências
13730
31


Tamanho do teste e quantidade de ocorrências
6900
13



/home/pdpa-chuvas/Deslizamento/env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
/tmp/ipykernel_261266/2109001105.py:6: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


/tmp/ipykernel_261266/2109001105.py:19: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


/tmp/ipykernel_261266/2109001105.py:32: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.




...

/home/pdpa-chuvas/Deslizamento/env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
/tmp/ipykernel_261266/2109001105.py:6: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.





Tamanho do treino e quantidade de ocorrências
253766
316



/tmp/ipykernel_261266/2109001105.py:19: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


Skipping line 66958: expected 46 fields, saw 51




Tamanho do teste e quantidade de ocorrências
221090


ValueError: cannot convert float NaN to integer

# Plot the threshold tuning
df_threshold_tuning = pd.DataFrame({'F-score':fscore,
                                    'Threshold':thresholds})
df_threshold_tuning.head()

plotnine.options.figure_size = (8, 4.8)
(
    ggplot(data = df_threshold_tuning)+
    geom_point(aes(x = 'Threshold',
                   y = 'F-score'),
               size = 0.4)+
    # Best threshold
    geom_point(aes(x = thresholdOpt,
                   y = fscoreOpt),
               color = '#981220',
               size = 4)+
    geom_line(aes(x = 'Threshold',
                   y = 'F-score'))+
    # Annotate the text
    geom_text(aes(x = thresholdOpt,
                  y = fscoreOpt),
              label = 'Optimal threshold \n for class: {}'.format(thresholdOpt),
              nudge_x = 0,
              nudge_y = -0.10,
              size = 10,
              fontstyle = 'italic')+
    labs(title = 'Threshold Tuning Curve')+
    xlab('Threshold')+
    ylab('F-score')+
    theme_minimal()
)